# WhisperX Pipeline v6

Пайплайн транскрибации видео уроков с диаризацией спикеров и LLM-анализом.

**Этапы:**
1. Загрузка видео с Яндекс Диска
2. Шумоподавление (noisereduce)
3. Транскрибация (WhisperX large-v3 + Silero VAD)
4. Пост-обработка + повторная транскрибация проблемных зон (Pass 2)
5. Выравнивание слов (wav2vec2)
6. Диаризация спикеров (pyannote 3.1)
7. Аналитика (шум, темп, баланс, вовлечённость)
8. LLM-анализ (GigaChat: таймлайн + оценка качества)

Перед запуском настройте секреты: **Add-ons → Secrets** → HF_TOKEN, GIGACHAT_CREDENTIALS, YANDEX_TOKEN

In [ ]:
import subprocess, shutil, os

# Удаляем старую копию репо если есть
REPO_DIR = "/kaggle/working/whisperx-pipeline"
if os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
    print("🗑️ Старая копия удалена")

# Установка зависимостей
print("📦 Установка пакетов...")
subprocess.run([
    "pip", "install", "-q",
    "whisperx", "faster-whisper", "pyannote.audio",
    "noisereduce", "yadisk", "gigachat", "pyyaml", "soundfile", "nltk"
], check=True)
print("✅ Пакеты установлены")

# Клонируем репозиторий
print("📥 Клонирование репозитория...")
subprocess.run(["git", "clone",
    "https://github.com/Hipposum/whisperx-pipeline.git",
    REPO_DIR], check=True)
print("✅ Репозиторий клонирован")

In [ ]:
from kaggle_secrets import UserSecretsClient
import os
s = UserSecretsClient()
os.environ["HF_TOKEN"] = s.get_secret("HF_TOKEN")
os.environ["GIGACHAT_CREDENTIALS"] = s.get_secret("GIGACHAT_CREDENTIALS")
os.environ["YANDEX_TOKEN"] = s.get_secret("YANDEX_TOKEN")

In [ ]:
import sys
sys.path.insert(0, "/kaggle/working/whisperx-pipeline")
from src.pipeline import main
main()